# ONNX to TensorFlow Lite Conversion Notebook

This notebook provides a standalone script to convert an ONNX model (e.g., `best.onnx`) to a TensorFlow Lite (`.tflite`) model, with an option for INT8 quantization.

**Prerequisites:**
- An ONNX model file (e.g., `best.onnx`).
- (Optional for INT8 quantization) A representative dataset for calibration. This notebook assumes the dataset is available locally or can be mounted from Google Drive.

**Notebook Structure:**
1.  **Setup:** Installs necessary libraries.
2.  **Configuration and Load ONNX Model:** Defines paths and loads the ONNX model.
3.  **Convert to TensorFlow Lite:** Converts the ONNX model to a TensorFlow SavedModel, then to TFLite, with optional INT8 quantization.
4.  **Save TFLite Model:** Saves the generated TFLite model.

In [ ]:
# Cell 1: Setup
# Install necessary libraries first
!pip install onnx tensorflow onnx_tf numpy tensorflow-addons==0.16.1

import os
import tensorflow as tf
import onnx
from onnx_tf.converter import convert as onnx_to_tf_convert
import numpy as np

In [ ]:
# Cell 2: Configuration and Load ONNX Model

# --- Configuration ---
# Path to your input ONNX model
onnx_model_path = "/content/drive/MyDrive/PawikanSentinel/models/best.onnx" # Adjust this path as needed

# Output path for the TFLite model
tflite_output_path = "/content/drive/MyDrive/PawikanSentinel/models/best-float32.tflite" # Adjust this path as needed

# --- For INT8 Quantization (Optional) ---
# Set to True to enable INT8 quantization, False for float32 TFLite model
enable_int8_quantization = False # Set to True to enable INT8 quantization

# Path to your dataset for calibration (required for INT8 quantization)
# This should point to a directory containing images that represent your typical input data.
# Example: "/content/datasets/pawikan_dataset/images/val"
representative_dataset_path = "/content/datasets/pawikan_dataset/images/val" # Adjust this path if using INT8

# Mount Google Drive if your models or datasets are there
from google.colab import drive
drive.mount('/content/drive')

# Load the ONNX model
print(f"Loading ONNX model from: {onnx_model_path}")
onnx_model = onnx.load(onnx_model_path)
print("ONNX model loaded successfully.")

In [ ]:
# Cell 3: Convert to TensorFlow Lite

print("Converting ONNX model to TensorFlow Lite...")

# Convert ONNX model to TensorFlow SavedModel
tf_model_path = "/tmp/tf_model" # Temporary path to save TensorFlow SavedModel
onnx_to_tf_convert(onnx_model, tf_model_path)
print(f"ONNX model converted to TensorFlow SavedModel at: {tf_model_path}")

# Convert TensorFlow SavedModel to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_saved_model(tf_model_path)

if enable_int8_quantization:
    print("Enabling INT8 quantization...")
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8  # Or tf.uint8
    converter.inference_output_type = tf.int8 # Or tf.uint8

    # Define a representative dataset generator for calibration
    def representative_dataset():
        if not os.path.exists(representative_dataset_path):
            raise FileNotFoundError(f"Representative dataset path not found: {representative_dataset_path}")

        # Assuming images are in a directory and need to be preprocessed to model input shape
        # Replace this with your actual data loading and preprocessing logic
        image_files = [os.path.join(representative_dataset_path, f) for f in os.listdir(representative_dataset_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
        for i, image_file in enumerate(image_files):
            if i >= 100: # Use a subset of data for calibration (e.g., 100 images)
                break
            img = tf.io.read_file(image_file)
            img = tf.image.decode_image(img, channels=3) # Decode to 3 channels
            img = tf.image.resize(img, (640, 640)) # Resize to model input size (adjust if different)
            img = img / 255.0 # Normalize to [0, 1] (adjust if your model expects different range)
            img = tf.cast(img, tf.float32) # Cast to float32
            img = np.expand_dims(img, axis=0) # Add batch dimension
            yield [img]

    converter.representative_dataset = representative_dataset
    print("Representative dataset configured for INT8 quantization.")
else:
    print("Converting to float32 TFLite model (no quantization).")

tflite_model = converter.convert()
print("Conversion to TFLite complete.")

In [ ]:
# Cell 4: Save TFLite Model

print(f"Saving TFLite model to: {tflite_output_path}")
with open(tflite_output_path, 'wb') as f:
    f.write(tflite_model)
print("TFLite model saved successfully.")

# Verify file size
print(f"TFLite model size: {os.path.getsize(tflite_output_path) / (1024 * 1024):.2f} MB")